# AP4 - Treinamento Word2Vec com os dados dos artigos
# Processamento de Linguagem Natural com Word2Vec – STIL 2021

## AP4 - Word2Vec com dados já processados (Stanza)
## Sentenças extraídas do artigo_completo

## Bibliotecas instaladas:

| Biblioteca | Finalidade |
|------------|------------|
| `gensim` | Treinamento do Word2Vec |
| `nltk` | Tokenização de sentenças (Punkt) |
| `pandas` | Manipulação e exportação de resultados |
| `numpy` | Operações com vetores |
| `huggingface_hub` | Huggingface Hub |
| `safetensors` | Huggingface tensores |

In [5]:
%pip install -q gensim nltk pandas numpy huggingface_hub safetensors

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importação das bibliotecas

| Biblioteca/Módulo | Importação | Finalidade |
|-------------------|------------|-------------|
| `json` | `import json` | Leitura/escrita de arquivos JSON (dataset) |
| `numpy` | `import numpy as np` | Operações matemáticas com vetores |
| `pandas` | `import pandas as pd` | Criação de tabelas e exportação CSV |
| `collections.Counter` | `from collections import Counter` | Contagem de frequência de palavras |
| `gensim.models.Word2Vec` | `from gensim.models import Word2Vec` | Treinamento do modelo Word2Vec |
| `nltk` | `import nltk` | Download dos recursos do NLTK |
| `nltk.tokenize.sent_tokenize` | `from nltk.tokenize import sent_tokenize` | Segmentação de texto em sentenças |

### Observações:

| Item | Detalhe |
|------|---------|
| `sent_tokenize` | Requer `nltk.download('punkt')` e `nltk.download('punkt_tab')` |
| `Word2Vec` | Parâmetro `sg=1` para algoritmo skip-gram |
| `Counter` | Usado para encontrar substantivo/verbo mais frequente |

In [6]:
import json
import numpy as np
from safetensors.numpy import load_file
import pandas as pd
from collections import Counter
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import sent_tokenize
import os

### `punkt`
- **O que é:** Um **tokenizer** do NLTK que divide texto em **sentenças** e **palavras**
- **Para que serve:** Segmentar frases para treinar o Word2Vec (cada sentença é uma sequência de treinamento)
- **Exemplo:** `"Olá mundo! Como está?"` => `["Olá mundo!", "Como está?"]`

### `punkt_tab`
- **O que é:** Uma **versão alternativa/atualizada** do tokenizer Punkt
- **Para que serve:** Necessário em **versões mais novas do NLTK** (≥ 3.8.1)
- **Quando usar:** Para garantir compatibilidade com diferentes versões

In [7]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\Rafael
[nltk_data]     Brandão\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Rafael
[nltk_data]     Brandão\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Carregar dataset

Carregar o dataset do arquivo JSON já processado pelo Stanza
Sentenças extraídas do artigo_completo

In [8]:
with open('../output/dataset.json', 'r', encoding='utf-8') as f:
    articles = json.load(f)

## Função para alinhar lemas com setenças

Usa o artigo_completo do artigos processados no dataset para segmentar sentenças e alinhar os lemas processados.

Retorna lista de sentenças lematizadas.

In [9]:
def align_lemas_by_sentence(article):
    # Obter o texto completo do artigo
    full_article = article.get('artigo_completo', '')

    if not full_article:
        return []

    # Segmentar sentenças do artigo completo
    sentences = sent_tokenize(full_article, language='portuguese')
    
    # Obter tokens e lemas já processados (usando os nomes corretos)
    tokens = article.get('artigo_tokenizado', []) 
    lemmas = article.get('lema', [])
    
    if not tokens or not lemmas:
        return []
    
    # Alinhar: reconstruir sentenças a partir dos tokens
    lemmatized_sentences = []
    current_sentence = []

    for i, token in enumerate(tokens):
        # Adicionar lemma (se não for pontuação)
        if token not in ['.', '!', '?', ',', ';', ':']:
            if i < len(lemmas) and lemmas[i] is not None:
                current_sentence.append(lemmas[i].lower())
        
        # Se encontrar fim de sentença
        if token in ['.', '!', '?'] and current_sentence:
            lemmatized_sentences.append(current_sentence)
            current_sentence = []
    
    return lemmatized_sentences

## 3. Processar todos os artigos no dataset

In [10]:
print("Processando artigos e alinhando lemas por sentença...")

all_lemmatized_sentences = []
all_tokens = []
all_pos_tags = []
all_lemmas = []

for i, article in enumerate(articles):
    print(f"Artigo {i+1}: {article.get('titulo', 'Sem título')[:50]}...")
    
    # Extrair sentenças lematizadas
    sentences = align_lemas_by_sentence(article)
    all_lemmatized_sentences.extend(sentences)
    
    # Coletar estatísticas
    if 'artigo_tokenizado' in article:
        all_tokens.extend(article['artigo_tokenizado'])
    if 'pos_tagger' in article:
        all_pos_tags.extend(article['pos_tagger'])
    if 'lema' in article:
        all_lemmas.extend(article['lema'])
    
    print(f"  Sentenças extraídas: {len(sentences)}")

print(f"\nTotal de sentenças lematizadas: {len(all_lemmatized_sentences)}")
if all_lemmatized_sentences:
    print(f"Exemplo de sentença: {all_lemmatized_sentences[0][:10]}")

Processando artigos e alinhando lemas por sentença...
Artigo 1: Utilizando um dicionário morfológico para expandir...
  Sentenças extraídas: 168
Artigo 2: Explorando a revisão de corpora por meio da compar...
  Sentenças extraídas: 211
Artigo 3: PetroGold – Corpus padrão ouro para o domínio do p...
  Sentenças extraídas: 175
Artigo 4: Lexicalidade biomédica e sua mensuração em um corp...
  Sentenças extraídas: 138
Artigo 5: Análise de polaridade e de tópicos em tweets no do...
  Sentenças extraídas: 175
Artigo 6: Utilizando BERTimbau para a Classificação de Emoçõ...
  Sentenças extraídas: 144
Artigo 7: Classificação multimodal para detecção de produtos...
  Sentenças extraídas: 165
Artigo 8: DP-Symptom-Identifier: uma estratégia para classif...
  Sentenças extraídas: 164
Artigo 9: Identificando sintomas de depressão em postagens d...
  Sentenças extraídas: 173
Artigo 10: Detecção de desinformação sobre Covid-19 no Twitte...
  Sentenças extraídas: 176
Artigo 11: ReVera Framework: Um Fra

## 4. Treinar Word2Vec (Skip-gram)

In [11]:
print("\nTreinando Word2Vec com skip-gram...")
model_w2v = Word2Vec(
    sentences=all_lemmatized_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1, # skip-gram
    workers=4,
    epochs=10,
    seed=42
)

print(f"Vocabulário treinado: {len(model_w2v.wv)} palavras")


Treinando Word2Vec com skip-gram...
Vocabulário treinado: 10464 palavras


## 5. Estatísticas de frequência (usando POS tags)

In [12]:
frequency_by_pos_tags = {'NOUN': Counter(), 'VERB': Counter()}
freq_lemmas = Counter()

for i, lemma in enumerate(all_lemmas):
    if i < len(all_pos_tags):
        pos = all_pos_tags[i]

        if lemma is None:
            continue

        lemma_lower = lemma.lower()
        freq_lemmas[lemma_lower] += 1
        
        if pos in ['NOUN', 'PROPN']:
            frequency_by_pos_tags['NOUN'][lemma_lower] += 1
        elif pos == 'VERB':
            frequency_by_pos_tags['VERB'][lemma_lower] += 1

# Substantivo e verbo mais frequentes
most_frequent_noun = frequency_by_pos_tags['NOUN'].most_common(1)[0][0] if frequency_by_pos_tags['NOUN'] else "N/A"
most_frequent_verb = frequency_by_pos_tags['VERB'].most_common(1)[0][0] if frequency_by_pos_tags['VERB'] else "N/A"

print("\n" + "="*50)
print("RESPOSTAS - ATIVIDADE 1")
print("="*50)

print(f"a) Vetor para 'modelos': {model_w2v.wv['modelos'][:6] if 'modelos' in model_w2v.wv else 'N/A'}...")
print(f"b) Vetor para 'linguagem': {model_w2v.wv['linguagem'][:6] if 'linguagem' in model_w2v.wv else 'N/A'}...")
print(f"c) Substantivo de maior frequência: '{most_frequent_noun}'")
print(f"d) Verbo de maior frequência: '{most_frequent_verb}'")


RESPOSTAS - ATIVIDADE 1
a) Vetor para 'modelos': [ 0.02985627  0.0007155  -0.01468287 -0.02667382 -0.02078803  0.01210597]...
b) Vetor para 'linguagem': [-0.02732228  0.03920972  0.24107409 -0.29901242  0.11782354 -0.02949362]...
c) Substantivo de maior frequência: 'and'
d) Verbo de maior frequência: 'poder'


## 6. Termos mais similares

In [13]:
print("\n" + "="*50)
print("RESPOSTAS - ATIVIDADE 2")
print("="*50)

def show_similar(word, model, topn=5):
    try:
        similar = model.wv.most_similar(word, topn=topn)
        print(f"\nPalavras mais similares a '{word}':")
        
        for p, score in similar:
            print(f"  • {p} (score: {score:.4f})")
        return similar
    except KeyError:
        print(f"\nPalavra '{word}' não está no vocabulário")
        return []

print("\na) Termos similares a 'modelos':")
show_similar('modelos', model_w2v)

print("\nb) Termos similares a 'linguagem':")
show_similar('linguagem', model_w2v)

print(f"\nc) Termos similares a '{most_frequent_noun}':")
show_similar(most_frequent_noun, model_w2v)

print(f"\nd) Termos similares a '{most_frequent_verb}':")
show_similar(most_frequent_verb, model_w2v)


RESPOSTAS - ATIVIDADE 2

a) Termos similares a 'modelos':

Palavras mais similares a 'modelos':
  • decodificar (score: 0.9905)
  • pós-processamento (score: 0.9897)
  • 350 (score: 0.9891)
  • system-test+dev (score: 0.9885)
  • 6.277 (score: 0.9883)

b) Termos similares a 'linguagem':

Palavras mais similares a 'linguagem':
  • processamento (score: 0.9363)
  • pln (score: 0.9072)
  • natural (score: 0.9050)
  • tecnologia (score: 0.8968)
  • livre (score: 0.8767)

c) Termos similares a 'and':

Palavras mais similares a 'and':
  • in (score: 0.8281)
  • conference (score: 0.8072)
  • elra (score: 0.8058)
  • eds (score: 0.7922)
  • editors (score: 0.7917)

d) Termos similares a 'poder':

Palavras mais similares a 'poder':
  • possível (score: 0.8964)
  • já (score: 0.8720)
  • também (score: 0.8708)
  • esse (score: 0.8644)
  • ao (score: 0.8631)


[('possível', 0.8963764309883118),
 ('já', 0.8719931244850159),
 ('também', 0.8708029985427856),
 ('esse', 0.8644381761550903),
 ('ao', 0.8631036281585693)]

## 7. Soma e subtração de vetores

vector_operation: realiza operação de soma e subtração de vetores
- positivos: lista de palavras para somar
- negativos: lista de palavras para subtrair


In [17]:
print("="*60)
print("PARTE 7: SOMA E SUBTRAÇÃO DE VETORES")
print("="*60)

# 1. Carregar modelo pré-treinado NILC (arquivos baixados manualmente)
print("\n1. Carregando modelo NILC Skip-gram 100d (manual)...")

# Caminhos para os arquivos baixados
base_path = '../data/nilc-nlp'  # Ajuste conforme necessário
vector_path = os.path.join(base_path, 'embeddings.safetensors')
vocab_path = os.path.join(base_path, 'vocab.txt')

if os.path.exists(vector_path) and os.path.exists(vocab_path):
    try:
        print(f"  Vetores: {vector_path}")
        print(f"  Vocabulário: {vocab_path}")
        
        # Carregar a matriz de vetores
        print("  Carregando vetores...")
        data = load_file(vector_path)
        vectors = data["embeddings"]  # numpy array, shape: (vocab_size, 100)
        
        # Carregar o vocabulário
        print("  Carregando vocabulário...")
        with open(vocab_path, 'r', encoding='utf-8') as f:
            vocab = [linha.strip() for linha in f]
        
        print(f"  Arquivos carregados com sucesso!")
        print(f"     Shape da matriz: {vectors.shape}")
        print(f"     Tamanho do vocabulário: {len(vocab):,} palavras")
        
        # Converter para formato gensim (para usar most_similar)
        print("\n  Convertendo para formato gensim...")
        model_pretrained = KeyedVectors(vector_size=vectors.shape[1])
        
        # Adicionar vetores em batches (evita problemas de memória)
        model_pretrained.add_vectors(vocab, vectors)
        
        print(f"  Modelo NILC pronto para uso!")
        print(f"     Dimensão dos vetores: {model_pretrained.vector_size}")
        print(f"     Palavras no modelo: {len(model_pretrained.key_to_index):,}")
        
        # Teste rápido
        if 'modelo' in model_pretrained.key_to_index:
            print(f"\n  Teste: vetor de 'modelo' (primeiras 5 dims): {model_pretrained['modelo'][:5]}")
        
    except Exception as e:
        print(f"  Erro ao carregar: {e}")
        model_pretrained = None
else:
    print("  Arquivos não encontrados!")
    print(f"\n  Arquivo esperado (vetores): {vector_path}")
    print(f"  Arquivo esperado (vocab): {vocab_path}")
    model_pretrained = None

# 2. Seu modelo treinado com os artigos
print("\n2. Carregando seu modelo treinado...")

if 'model_w2v' in locals() and model_w2v:
    print(f"  Seu modelo carregado")
    print(f"     Vocabulário: {len(model_w2v.wv.key_to_index):,} palavras")
    print(f"     Dimensão: {model_w2v.wv.vector_size}")
else:
    print("  Modelo local não encontrado. Execute o treinamento primeiro.")
    model_w2v = None

# 3. Função de operação vetorial
# Executa most_similar com tratamento de erro
def vector_operation(model, positives, negatives, topn=5, nome=""):
    if model is None:
        return []
    try:
        return model.most_similar(positive=positives, negative=negatives, topn=topn)
    except KeyError as e:
        print(f"      {nome} - Palavra não encontrada: {e}")
        return []
    except Exception as e:
        print(f"      {nome} - Erro: {e}")
        return []

# 4. Comparação das operações
print("\n" + "-"*60)
print("3. COMPARAÇÃO DAS OPERAÇÕES VETORIAIS")
print("-"*60)

# Operação acadêmica 0: processamento - linguagem + texto (áreas de PLN)
print("\n  'processamento' + 'texto' - 'linguagem'")
print("     (Esperado: mineração, extração, análise de texto)")
print("  " + "="*40)

if model_pretrained:
    print("\n  MODELO PRÉ-TREINADO NILC (Skip-gram):")
    result = vector_operation(model_pretrained, ['processamento', 'texto'], ['linguagem'], topn=5, nome="NILC")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

if model_w2v:
    print("\n  SEU MODELO (treinado com artigos):")
    result = vector_operation(model_w2v.wv, ['processamento', 'texto'], ['linguagem'], topn=5, nome="Artigo")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

# Operação acadêmica 1: linguagem + dados - processamento
print("\n" + "-"*40)
print("  'linguagem' + 'dados' - 'processamento'")
print("  " + "="*40)

if model_pretrained:
    print("\n  MODELO PRÉ-TREINADO NILC:")
    result = vector_operation(model_pretrained, ['linguagem', 'dados'], ['processamento'], topn=5, nome="NILC")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

if model_w2v:
    print("\n  SEU MODELO (treinado com artigos):")
    result = vector_operation(model_w2v.wv, ['linguagem', 'dados'], ['processamento'], topn=5, nome="Artigo")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

# Operação acadêmica 2: artigo + pesquisa - linguagem
print("\n" + "-"*40)
print("  'artigo' + 'pesquisa' - 'linguagem'")
print("  " + "="*40)

if model_pretrained:
    print("\n  MODELO PRÉ-TREINADO NILC:")
    result = vector_operation(model_pretrained, ['artigo', 'pesquisa'], ['linguagem'], topn=5, nome="NILC")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

if model_w2v:
    print("\n  SEU MODELO (treinado com artigos):")
    result = vector_operation(model_w2v.wv, ['artigo', 'pesquisa'], ['linguagem'], topn=5, nome="Artigo")
    for i, (word, score) in enumerate(result, 1):
        print(f"     {i}. {word}: {score:.4f}")

# 5. Demonstração manual de soma/subtração
if model_pretrained:
    print("\n" + "-"*60)
    print("4. DEMONSTRAÇÃO MANUAL (cálculo vetorial)")
    print("-"*60)
    
    # Verificar se as palavras existem
    test_words = ['processamento', 'texto', 'linguagem']
    existing_words = [p for p in test_words if p in model_pretrained.key_to_index]
    
    if len(existing_words) == 3:
        v_processing = model_pretrained['processamento']
        v_text = model_pretrained['texto']
        v_language = model_pretrained['linguagem']
        
        # Calcular: rei - homem + mulher
        v_result = v_processing - v_text + v_language
        
        print("\n  Cálculo passo a passo (NILC):")
        print(f"    vetor('processamento')     = {v_processing[:3]}...")
        print(f"    vetor('texto')   = {v_text[:3]}...")
        print(f"    vetor('linguagem')  = {v_language[:3]}...")
        print(f"    vetor(resultado) = {v_result[:3]}...")
        
        # Encontrar palavra mais similar
        similar = model_pretrained.similar_by_vector(v_result, topn=3)
        print(f"\n  Palavras similares ao vetor resultante:")
        for i, (word, score) in enumerate(similar, 1):
            print(f"    {i}. {word}: {score:.4f}")

# 6. Resumo final
print("\n" + "="*60)
print("5. RESUMO DOS MODELOS")
print("="*60)

if model_pretrained:
    print(f"""
  NILC Skip-gram 100d (pré-treinado)
     • Arquitetura: Skip-gram
     • Dimensão: 100
     • Vocabulário: {len(model_pretrained.key_to_index):,} palavras
     • Corpus: 1.39B tokens (17 fontes diversas)
     • Cobertura: Conhecimento geral do português
    """)

if model_w2v:
    print(f"""
  Seu modelo (treinado com artigos)
     • Arquitetura: Skip-gram
     • Dimensão: {model_w2v.wv.vector_size}
     • Vocabulário: {len(model_w2v.wv.key_to_index):,} palavras
     • Corpus: {len(articles)} artigos acadêmicos de PLN
     • Cobertura: Conhecimento específico do domínio
    """)

print("="*60)

PARTE 7: SOMA E SUBTRAÇÃO DE VETORES

1. Carregando modelo NILC Skip-gram 100d (manual)...
  Vetores: ../data/nilc-nlp\embeddings.safetensors
  Vocabulário: ../data/nilc-nlp\vocab.txt
  Carregando vetores...
  Carregando vocabulário...
  Arquivos carregados com sucesso!
     Shape da matriz: (929606, 100)
     Tamanho do vocabulário: 929,606 palavras

  Convertendo para formato gensim...
  Modelo NILC pronto para uso!
     Dimensão dos vetores: 100
     Palavras no modelo: 929,603

  Teste: vetor de 'modelo' (primeiras 5 dims): [-0.050928 -0.430445  0.10267  -0.160677  0.04497 ]

2. Carregando seu modelo treinado...
  Seu modelo carregado
     Vocabulário: 10,464 palavras
     Dimensão: 100

------------------------------------------------------------
3. COMPARAÇÃO DAS OPERAÇÕES VETORIAIS
------------------------------------------------------------

  'processamento' + 'texto' - 'linguagem'
     (Esperado: mineração, extração, análise de texto)

  MODELO PRÉ-TREINADO NILC (Skip-gram):
